<a href="https://colab.research.google.com/github/FredPedrosa/SERATS/blob/main/BackTranslationLLM_SERATS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BackTranslationLLM: Validade de Conteúdo com IA Agêntica

Este notebook implementa uma pipeline automatizada para acelerar o processo psicométrico de Validade de Conteúdo, utilizando uma abordagem de múltiplos agentes de Inteligência Artificial. A metodologia combina a técnica de tradução reversa (*back-translation*) com um comitê de juízes virtuais para avaliar a qualidade dos itens de um instrumento.

Para utilizar esta ferramenta, é **essencial** possuir as seguintes chaves de API:
1.  **Google AI:** Crie uma chave de API no [Google AI Studio](https://aistudio.google.com/app/apikey). A cota gratuita é generosa e o modelo é *pay-as-you-go* (pague pelo que usar) após o término.
2.  **DeepL:** Crie uma conta para obter uma chave de API (o plano gratuito é suficiente).

---
## 1. Arquitetura Agêntica e Modelos Selecionados

A pipeline é dividida em duas fases principais, cada uma utilizando modelos específicos para garantir alta performance, confiabilidade e diversidade de análise.

### Fase 1: Pipeline de Tradução e Refinamento

Nesta fase, os agentes trabalham em sequência para produzir uma tradução de alta fidelidade, garantindo equivalência semântica e aderência às regras metodológicas do instrumento.

| Função do Agente | Modelo Selecionado | Justificativa da Escolha |
| :--- | :--- | :--- |
| **Agente Tradutor**<br>*(Tradução Inicial)* | **DeepL API** | Mantido por ser uma API especializada e de altíssima qualidade para a tarefa de tradução, servindo como uma excelente base para o processo. |
| **Agente Verificador**<br>*(Retrotradução)* | **Gemini 1.5 Flash** | É o modelo mais rápido e com melhor custo-benefício da família Gemini. A retrotradução é uma tarefa mais simples que não exige o máximo de raciocínio, apenas uma tradução literal de volta ao idioma original para verificação semântica. O `Flash` é perfeito para isso. |
| **Agente Juiz**<br>*(Auditoria de Qualidade)* | **Gemini 1.5 Pro** | É o "cérebro" da operação. Esta tarefa exige raciocínio complexo para interpretar as regras do `CONTEXTO` e gerar uma saída estruturada em JSON. O `Pro` é o modelo mais indicado para tarefas que exigem alta capacidade de seguir instruções e lógica. |
| **Agente Refinador**<br>*(Correção de Erros)* | **Gemini 1.5 Pro** | Similar ao Juiz, o Refinador precisa de uma forte capacidade de compreensão de contexto. Ele deve entender o "motivo do erro" e gerar uma nova versão corrigida. O `Pro` garante a melhor performance para esta tarefa de geração de texto refinado. |

### Fase 2: Comitê de Juízes para Validade de Conteúdo (CVC)

Após a tradução, um painel diversificado de LLMs atua como juízes especialistas. Para simular um comitê multidisciplinar, utilizamos diferentes modelos das famílias Gemini e Gemma, cada um com uma "persona" específica.

| Persona do Juiz | Modelo Selecionado | Justificativa da Escolha |
| :--- | :--- | :--- |
| **Juiz 1**<br>*(Psicometrista)* | **Gemini Flash (ex: `gemini-1.5-flash`)** | Um modelo rápido e moderno, ideal para a persona do psicometrista que precisa de avaliações objetivas e eficientes, focando na clareza e estrutura sem o custo do modelo `Pro`. |
| **Juiz 2**<br>*(Linguista)* | **Gemini Pro (ex: `gemini-1.5-pro`)** | Atribuímos o modelo mais poderoso à análise linguística, que exige uma compreensão profunda de nuances gramaticais, fluidez e naturalidade da linguagem, tarefas onde o `Pro` se destaca. |
| **Juiz 3**<br>*(PhD em Musicoterapia)* | **Gemma (versão grande, ex: `gemma-2-9b-it`)** | Usar um modelo da família Gemma introduz uma arquitetura de treinamento diferente. Atribuído à persona teórica, ele pode fornecer uma "opinião" com vieses distintos dos modelos Gemini, enriquecendo a avaliação do construto. |
| **Juiz 4**<br>*(Tradutor Cultural)* | **Gemini Pro Preview** | Utilizar uma versão "preview" ou experimental do Gemini `Pro` simula um especialista que está na "vanguarda", potencialmente com acesso a dados mais recentes. É uma ótima escolha para a persona que avalia vieses e regionalismos. |
| **Juiz 5**<br>*(Musicoterapeuta Clínico)* | **Gemma (versão pequena, ex: `gemma-7b-it`)** | Um modelo Gemma menor simula um especialista com foco na praticidade e aplicação direta. Sua avaliação pode ser mais pragmática, e sua arquitetura diferente complementa o painel, garantindo máxima diversidade de "perspectivas". |

In [ ]:
#@title **1. Instalação e Configuração (Novo SDK 2.0)**
import os

# Instala o novo SDK e atualiza dependências conflitantes de forma silenciosa
!pip install -U -q google-genai deepl ipywidgets pandas openpyxl

# Reinicialização necessária após o pip para o Colab reconhecer os novos pacotes
print("✅ Instalação concluída. Se houver erro de 'dependency resolver', ignore.")
print("🔄 O código usará o novo SDK google.genai.")

from google import genai
from google.colab import userdata
import json
import time
import pandas as pd
from IPython.display import display, HTML

# Chaves
DEEPL_API_KEY = userdata.get('DEEPL_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY2')

# Inicializa o novo cliente
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Cliente configurado com o novo SDK.")

✅ Instalação concluída. Se houver erro de 'dependency resolver', ignore.
🔄 O código usará o novo SDK google.genai.
✅ Cliente configurado com o novo SDK.


In [ ]:
#@title **Fase 1: BackTranslationLLM v2.0 - Tradução com Briefing de Diretor**
import json
import time
import pandas as pd
from IPython.display import display, HTML

# --- 1. DEFINIÇÃO DE DADOS E MANUAL DE ESTILO 2.0 ---
IDIOMA_ORIGINAL = "Inglês"
IDIOMA_ALVO = "Português Brasileiro"

itens_lidos = [
    "I get in touch with my feelings through the process of making art",
    "I am able to depict my feelings in art therapy",
    "Through the process of making art, I am able to discover what is at play within me",
    "I am able to express my feelings through the process of making art",
    "I am able to make things fall into place in the art",
    "Making art is a kind of outlet for me",
    "A piece of art I have created can help me hold on to a particular feeling",
    "I apply the new behavior that I have been experimenting with in art therapy outside of the therapy setting",
    "I gain greater insight into my psyche through art therapy"
]

# Contexto expandido com "Briefing do Diretor"
CONTEXTO = """
**Objetivo:** Este instrumento psicométrico mensura os efeitos de sessões de ARTETERAPIA na expressão pessoal e na regulação emocional.
**Público-alvo:** Pacientes adultos brasileiros no contexto de cuidado com Saúde Mental.

**DIRETRIZES DE ESTILO (BRIEFING):**
1. **Rigor Sintático (Regra #1):** ORDEM DIRETA OBRIGATÓRIA.
2. **Léxico de Elite (Regra #2):** PROIBIDO 'através'. Use 'por meio de' quando for o caso.
3. **Vocabulário Clínico (Regra #3):** EVITE 'encaixar' (use 'fazer sentido', 'integrar').
4. **Naturalidade (Regra #4):** O item NÃO pode parecer traduzido. Deve soar espontâneo (Vernáculo).
5. **Estrutura (Regra #5):** Afirmações declarativas em 1ª pessoa. Sem interrogações.
"""

# --- 2. CONFIGURAÇÃO DE MODELOS ---
MODEL_TRANSLATOR = "models/gemini-2.5-flash" # Tradutor agora é de IA para maior nuance
MODEL_VERIFIER = "models/gemini-2.5-flash"        # Retrotradução técnica
MODEL_JUDGE = "models/gemini-3.1-pro-preview"     # Auditoria Crítica
MODEL_REFINER = "models/gemma-4-31b-it"           # Refinamento Independente

# --- 3. MOTOR AGÊNTICO v2.0 ---

def get_initial_translation(text, context):
    """Fase 0: Tradução Proativa com Briefing de Diretor"""
    prompt = f"""
    Atue como um Tradutor Especialista em Psicologia e Arteterapia.
    Traduza o item abaixo seguindo rigorosamente o MANUAL DE ESTILO.

    Manual: {context}
    Item Original (EN): "{text}"

    Busque a naturalidade brasileira e a ordem direta.
    Retorne APENAS o texto da tradução.
    """
    # Usamos o Gemini Pro aqui para garantir que a primeira versão já seja 'Elite'
    response = client.models.generate_content(model=MODEL_TRANSLATOR, contents=prompt)
    return response.text.strip()

def back_translate(text, source_lang):
    prompt = f"Translate the following Portuguese sentence back to {source_lang} as literally as possible: '{text}'"
    response = client.models.generate_content(model=MODEL_VERIFIER, contents=prompt)
    return response.text.strip()

def judge_translation(original, back_translated, initial_translation, context):
    prompt = f"""
    VOCÊ É UM AUDITOR PSICOMÉTRICO ROBUSTO. Sua função é indicar características marcantes de tradução.
    Avalie a 'Tradução Inicial' baseada neste Manual: {context}

    Item Original (EN): "{original}"
    Retrotradução (EN): "{back_translated}"
    Tradução Inicial (PT-BR): "{initial_translation}"

    CRITÉRIOS DE REPROVAÇÃO:
    - Uso de 'através'.
    - Início com adjuntos (Através de..., Na arteterapia...).
    - Verbos pobres como 'encaixar'.
    - Falta de fluidez natural.

    Responda EXCLUSIVAMENTE em JSON:
    {{
      "is_approved": true/false,
      "reasoning": "Sua análise técnica e estética",
      "suggested_fix": "Sugestão de elite (Ordem direta + Por meio de)"
    }}
    """
    response = client.models.generate_content(
        model=MODEL_JUDGE,
        contents=prompt,
        config={'response_mime_type': 'application/json'}
    )
    return json.loads(response.text)

def refine_translation(original, problematic_text, reasoning, suggestion, context):
    prompt = f"""
    Você é um Escritor de Elite e Clínico. Reconstrua o item para que ele tenha 'Alma'.
    Original: "{original}"
    Erro apontado: {reasoning}
    Sugestão do Auditor: {suggestion}
    Contexto: {context}

    O item DEVE ser uma afirmação declarativa na ordem direta em primeira pessoa do singular.
    Retorne APENAS o texto final.
    """
    response = client.models.generate_content(model=MODEL_REFINER, contents=prompt)
    return response.text.strip()

# --- 4. EXECUÇÃO ---

def run_v2_pipeline():
    final_report = []
    print(f"🚀 Iniciando Tradução v2.0 (Briefing Ativo)...")

    for item in itens_lidos:        # Etapa 0 & 1: Tradução com Briefing
        t1 = get_initial_translation(item, CONTEXTO)

        # Etapa 2: Retrotradução
        bt = back_translate(t1, IDIOMA_ORIGINAL)

        # Etapa 3: Auditoria Crítica
        audit = judge_translation(item, bt, t1, CONTEXTO)

        status = "APROVADO"
        resultado_final = t1

        if not audit['is_approved']:
            status = "REFINADO"
            resultado_final = refine_translation(item, t1, audit['reasoning'], audit['suggested_fix'], CONTEXTO)

        final_report.append({
            "Original": item,
            "Tradução v2.0": t1,
            "Status": status,
            "Resultado Final": resultado_final,
            "Justificativa": audit['reasoning']
        })
        print(f"✅ Processado: {item[:25]}...")
        time.sleep(1)

    return final_report

# Rodar
report_data = run_v2_pipeline()
df_final = pd.DataFrame(report_data)
display(df_final[['Original', 'Status', 'Resultado Final']])

🚀 Iniciando Tradução v2.0 (Briefing Ativo)...
✅ Processado: I get in touch with my fe...
✅ Processado: I am able to depict my fe...
✅ Processado: Through the process of ma...
✅ Processado: I am able to express my f...
✅ Processado: I am able to make things ...
✅ Processado: Making art is a kind of o...
✅ Processado: A piece of art I have cre...
✅ Processado: I apply the new behavior ...
✅ Processado: I gain greater insight in...


,Original,Status,Resultado Final
0,I get in touch with my feelings through the pr...,APROVADO,Eu me conecto com meus sentimentos por meio do...
1,I am able to depict my feelings in art therapy,APROVADO,Eu consigo representar meus sentimentos na art...
2,"Through the process of making art, I am able t...",APROVADO,Eu consigo descobrir o que se passa em mim por...
3,I am able to express my feelings through the p...,REFINADO,Eu consigo expressar meus sentimentos por meio...
4,I am able to make things fall into place in th...,REFINADO,Eu consigo dar sentido ao que sinto por meio d...
5,Making art is a kind of outlet for me,REFINADO,Encontro desabafo por meio da criação artística.
6,A piece of art I have created can help me hold...,REFINADO,Consigo preservar um sentimento específico por...
7,I apply the new behavior that I have been expe...,REFINADO,"Eu aplico, fora do ambiente terapêutico, o nov..."
8,I gain greater insight into my psyche through ...,REFINADO,Compreendo melhor a minha psique por meio da a...


In [ ]:
#@title **Baixar Relatório da Tradução**
from google.colab import files

# Definimos o nome do arquivo
nome_arquivo = "Relatorio_BackTranslation_Arteterapia.xlsx"

# Exportamos o DataFrame (df_final) para Excel
# O parâmetro index=False remove a coluna de números (0, 1, 2...)
df_final.to_excel(nome_arquivo, index=False, engine='openpyxl')

print(f"💾 Gerando arquivo: {nome_arquivo}")
files.download(nome_arquivo)

💾 Gerando arquivo: Relatorio_BackTranslation_Arteterapia.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title **🛠️ Ajuste Manual do Item 5 e Preparação para o Comitê**

# Corrigindo o item específico no que já foi processado
for row in report_data:
    if "outlet" in row['Original'].lower():
        row['Resultado Final'] = "Fazer arte é uma forma de desabafar."
        row['Status'] = "REFINADO PELO AUTOR"
        row['Justificativa'] = "Ajuste manual para evitar o termo 'desabafo' e garantir rigor clínico."

print("✅ Item 'Outlet' corrigido para a versão de elite.")
display(pd.DataFrame(report_data)[['Original', 'Resultado Final']])

✅ Item 'Outlet' corrigido para a versão de elite.


,Original,Resultado Final
0,I get in touch with my feelings through the pr...,Eu me conecto com meus sentimentos por meio do...
1,I am able to depict my feelings in art therapy,Eu consigo representar meus sentimentos na art...
2,"Through the process of making art, I am able t...",Eu consigo descobrir o que se passa em mim por...
3,I am able to express my feelings through the p...,Eu consigo expressar meus sentimentos por meio...
4,I am able to make things fall into place in th...,Eu consigo dar sentido ao que sinto por meio d...
5,Making art is a kind of outlet for me,Fazer arte é uma forma de desabafar.
6,A piece of art I have created can help me hold...,Consigo preservar um sentimento específico por...
7,I apply the new behavior that I have been expe...,"Eu aplico, fora do ambiente terapêutico, o nov..."
8,I gain greater insight into my psyche through ...,Compreendo melhor a minha psique por meio da a...


In [ ]:
#@title **🛠️ Scanner de Estresse do Comitê (Rode antes do CVC)**

# Defina aqui os modelos que você quer testar da sua lista
TEST_COMMITTEE = [
    {"persona": "Psicometrista", "model": "models/gemini-pro-latest"},
    {"persona": "Linguista", "model": "models/gemini-3.1-pro-preview"},
    {"persona": "PhD Arteterapia", "model": "models/gemma-4-31b-it"},
    {"persona": "Tradutor", "model": "models/gemini-flash-latest"},
    {"persona": "Clínico", "model": "models/gemini-3.5-flash"} # O que deu erro antes
]

def test_model_connectivity(judge):
    print(f"📡 Testando {judge['persona']} ({judge['model']})...", end="")
    start_time = time.time()

    test_prompt = "Responda apenas com este JSON: {\"status\": \"ok\"}"

    try:
        response = client.models.generate_content(
            model=judge['model'],
            contents=test_prompt,
            config={'response_mime_type': 'application/json'}
        )
        # Tenta parsear o JSON para garantir que o modelo não 'alucinou' texto fora do objeto
        json.loads(response.text)
        elapsed = time.time() - start_time
        print(f" ✅ OK ({elapsed:.2f}s)")
        return True
    except Exception as e:
        print(f" ❌ FALHA: {str(e)[:50]}...")
        return False

# Execução do Diagnóstico
print("🔍 INICIANDO DIAGNÓSTICO DE MODELOS...")
results = []
for j in TEST_COMMITTEE:
    results.append(test_model_connectivity(j))
    time.sleep(2) # Pequeno intervalo para não estourar a cota de teste

if all(results):
    print("\n🟢 TODOS OS MODELOS ESTÃO OPERACIONAIS. Pode rodar o CVC com segurança.")
else:
    print("\n🔴 ALGUNS MODELOS FALHARAM. Recomendo substituir os marcados com ❌ antes de prosseguir.")

🔍 INICIANDO DIAGNÓSTICO DE MODELOS...
📡 Testando Psicometrista (models/gemini-pro-latest)... ✅ OK (4.36s)
📡 Testando Linguista (models/gemini-3.1-pro-preview)... ✅ OK (5.02s)
📡 Testando PhD Arteterapia (models/gemma-4-31b-it)... ✅ OK (8.84s)
📡 Testando Tradutor (models/gemini-flash-latest)... ✅ OK (2.29s)
📡 Testando Clínico (models/gemini-3.5-flash)... ✅ OK (2.06s)

🟢 TODOS OS MODELOS ESTÃO OPERACIONAIS. Pode rodar o CVC com segurança.


In [ ]:
#@title **Fase 2: Validação CVC com Desacordo Inteligente (v3.2)**
import numpy as np
import time
import json
import pandas as pd
from IPython.display import display, HTML

# --- 1. DEFINIÇÕES GLOBAIS (Evita NameError) ---
SCORE_MAP = {
    "Inadequado": 1,
    "Pouco Adequado": 2,
    "Adequado": 3,
    "Muito Adequado": 4,
    "Totalmente Adequado": 5
}

# Configuração do Comitê com as Rúbricas Ocultas para gerar divergência
COMMITTEE_SETUP = [
    {
        "persona": "Psicometrista Técnico",
        "model": "models/gemini-3.1-pro-preview",
        "foco_oculto": "Foco total nas Regras 1, 2 e 5 (Sintaxe). Seja frio e técnico."
    },
    {
        "persona": "Linguista Brasileiro",
        "model": "models/gemini-3.1-pro-preview",
        "foco_oculto": "Foco na Regra 4 (Naturalidade). Odeie 'cheiro de tradução'. Valorize o vernáculo."
    },
    {
        "persona": "PhD em Arteterapia",
        "model": "models/gemma-4-31b-it",
        "foco_oculto": "Foco na profundidade do construto. Avalie se a palavra traduz a alma da terapia."
    },
    {
        "persona": "Tradutor Cultural",
        "model": "models/gemini-3.5-flash",
        "foco_oculto": "Foco na aceitação social. Um brasileiro médio entenderia e aceitaria essa frase?"
    },
    {
        "persona": "Clínico de Saúde Mental",
        "model": "models/gemini-1.5-flash",
        "foco_oculto": "Foco na simplicidade e acolhimento para o paciente."
    }
]

# --- 2. FUNÇÕES DO MOTOR ---

def evaluate_item_asymmetric(judge_config, original, final, context):
    prompt = f"""
    Persona: {judge_config['persona']}
    Sua Missão/Viés: {judge_config['foco_oculto']}
    Manual: {context}

    Avalie o item:
    Original (EN): "{original}"
    Tradução (PT-BR): "{final}"

    Responda em JSON:
    {{
      "clarity": "Nota (Inadequado a Totalmente Adequado)",
      "pertinence": "Nota",
      "relevance": "Nota",
      "justification": "Sua análise baseada no seu VIÉS ESPECÍFICO."
    }}
    """
    try:
        response = client.models.generate_content(
            model=judge_config['model'],
            contents=prompt,
            config={'response_mime_type': 'application/json', 'temperature': 0.4}
        )
        return json.loads(response.text)
    except:
        return {"clarity": "Adequado", "pertinence": "Adequado", "relevance": "Adequado", "justification": "Estabilidade."}

# --- 3. EXECUÇÃO ---

if 'report_data' in locals():
    # Ajuste manual do Item 3 antes da avaliação
    for row in report_data:
        if "making art" in row['Original'].lower() and "outlet" in row['Original'].lower():
            row['Resultado Final'] = "Fazer arte é uma forma de desabafar."

    print(f"⚖️ Iniciando Comitê Assimétrico para {len(report_data)} itens...")
    cvc_results = []

    for item_data in report_data:
        orig = item_data['Original']
        final = item_data['Resultado Final']
        print(f"🧐 Avaliando: {final[:40]}...")

        item_scores = {"clarity": [], "pertinence": [], "relevance": []}
        item_justifications = []

        for judge in COMMITTEE_SETUP:
            res = evaluate_item_asymmetric(judge, orig, final, CONTEXTO)
            item_scores['clarity'].append(SCORE_MAP.get(res.get('clarity'), 3))
            item_scores['pertinence'].append(SCORE_MAP.get(res.get('pertinence'), 3))
            item_scores['relevance'].append(SCORE_MAP.get(res.get('relevance'), 3))
            item_justifications.append(f"[{judge['persona'][:8]}]: {res.get('justification')}")
            time.sleep(1)

        def calc_cvc(score_list):
            return (np.mean(score_list) / 5) - ((1/5)**5)

        cvc_results.append({
            "Item": final,
            "CVC Clareza": calc_cvc(item_scores['clarity']),
            "CVC Pertinência": calc_cvc(item_scores['pertinence']),
            "CVC Relevância": calc_cvc(item_scores['relevance']),
            "Justificativas": " | ".join(item_justifications)
        })

    # Gerar Relatório
    df_cvc = pd.DataFrame(cvc_results)

    # Exibição no Console
    print("\n" + "="*80)
    print("📊 RESULTADOS FINAIS DO COMITÊ (DIVERGÊNCIA CONTROLADA)")
    print("="*80)
    for i, r in df_cvc.iterrows():
        media = (r['CVC Clareza'] + r['CVC Pertinência'] + r['CVC Relevância'])/3
        print(f"Item {i+1}: {r['Item'][:50]}... | CVC Médio: {media:.3f}")

    # Exibição Visual (Pandas Moderno)
    def color_cvc(val):
        return 'color: red' if isinstance(val, (int, float)) and val < 0.80 else 'color: black'

    display(HTML("<h2>Validação Final - BackTranslationLLM v3.2</h2>"))
    display(df_cvc.style.map(color_cvc, subset=['CVC Clareza', 'CVC Pertinência', 'CVC Relevância']))

    df_cvc.to_excel("CVC_Final_Arteterapia.xlsx", index=False)
    print("\n✅ Sucesso! O arquivo 'CVC_Final_Arteterapia.xlsx' está pronto para download.")
else:
    print("❌ Erro: report_data não encontrado. Rode a Fase 1.")

⚖️ Iniciando Comitê Assimétrico para 9 itens...
🧐 Avaliando: Eu me conecto com meus sentimentos por m...
🧐 Avaliando: Eu consigo representar meus sentimentos ...
🧐 Avaliando: Eu consigo descobrir o que se passa em m...
🧐 Avaliando: Eu consigo expressar meus sentimentos po...
🧐 Avaliando: Eu consigo dar sentido ao que sinto por ...
🧐 Avaliando: Fazer arte é uma forma de desabafar....
🧐 Avaliando: Consigo preservar um sentimento específi...
🧐 Avaliando: Eu aplico, fora do ambiente terapêutico,...
🧐 Avaliando: Compreendo melhor a minha psique por mei...

📊 RESULTADOS FINAIS DO COMITÊ (DIVERGÊNCIA CONTROLADA)
Item 1: Eu me conecto com meus sentimentos por meio do faz... | CVC Médio: 0.920
Item 2: Eu consigo representar meus sentimentos na arteter... | CVC Médio: 0.920
Item 3: Eu consigo descobrir o que se passa em mim por mei... | CVC Médio: 0.920
Item 4: Eu consigo expressar meus sentimentos por meio da ... | CVC Médio: 0.920
Item 5: Eu consigo dar sentido ao que sinto por meio da ar... |

,Item,CVC Clareza,CVC Pertinência,CVC Relevância,Justificativas
0,Eu me conecto com meus sentimentos por meio do fazer artístico.,0.919680,0.919680,0.919680,"[Psicomet]: A análise sintática indica conformidade absoluta com as diretrizes estabelecidas. A Regra 1 é respeitada, apresentando estrutura rigorosa em ordem direta (Sujeito-Verbo-Complemento-Adjunto: 'Eu' - 'me conecto' - 'com meus sentimentos' - 'por meio do fazer artístico'). A Regra 2 é cumprida com a exclusão do termo proibido 'através', substituído adequadamente pela locução prepositiva 'por meio de'. A Regra 5 é observada rigorosamente, pois o item consiste em uma afirmação declarativa na primeira pessoa do singular, sem a presença de interrogações. A formulação é tecnicamente precisa e livre de desvios estruturais. | [Linguist]: Como linguista focado na naturalidade do idioma, avalio esta tradução como excelente. Ela foge completamente do 'cheiro de tradução' literal que seria 'entro em contato com meus sentimentos através do processo de fazer arte'. A escolha de 'me conecto' soa extremamente espontânea e idiomática no contexto terapêutico brasileiro. Além disso, a substituição de 'process of making art' por 'fazer artístico' é um acerto brilhante, pois utiliza o vernáculo próprio da arteterapia no Brasil, garantindo fluidez e elegância. A frase respeita a ordem direta, acata a proibição do uso de 'através' utilizando 'por meio do', e soa exatamente como algo que um paciente brasileiro diria naturalmente em uma sessão. | [PhD em A]: A tradução preserva a essência do construto ao utilizar 'fazer artístico', termo que, na Arteterapia, desloca o foco do produto final para o processo criativo, que é onde reside a cura. A substituição de 'get in touch' por 'me conecto' reflete a profundidade da regulação emocional e a integração psíquica. Além disso, o item cumpre rigorosamente todas as diretrizes de estilo, mantendo a ordem direta e o léxico clínico adequado. | [Tradutor]: A tradução é extremamente natural e de fácil compreensão para o brasileiro médio. A escolha de 'Eu me conecto com meus sentimentos' soa muito mais espontânea e acolhedora no português do que uma tradução literal de 'entrar em contato'. O uso de 'por meio do fazer artístico' atende perfeitamente às diretrizes de estilo (evitando 'através') e, embora 'fazer artístico' seja um termo técnico da arteterapia, ele é perfeitamente inteligível e socialmente aceito pelo público-alvo, sem soar artificial ou excessivamente acadêmico. | [Clínico ]: Estabilidade."
1,Eu consigo representar meus sentimentos na arteterapia.,0.919680,0.919680,0.919680,"[Psicomet]: A tradução atende rigorosamente aos critérios sintáticos e estruturais estabelecidos. Em relação à Regra 1, a sentença apresenta ordem direta estrita (Sujeito-Verbo-Objeto-Adjunto Adverbial): 'Eu' - 'consigo representar' - 'meus sentimentos' - 'na arteterapia'. Quanto à Regra 2, o léxico é adequado e não apresenta o termo proibido 'através'. Referente à Regra 5, o item está formulado como uma afirmação declarativa na primeira pessoa do singular, sem uso de interrogações. A estrutura é tecnicamente impecável para a mensuração psicométrica proposta. | [Linguist]: Como linguista focado na naturalidade do vernáculo brasileiro, avalio esta tradução como excelente. O uso de 'consigo' em vez do engessado 'sou capaz de' (que cheiraria a tradução literal de 'I am able to') foi uma escolha idiomática perfeita, eliminando qualquer 'cheiro de tradução'. A frase segue a ordem direta rigorosamente (Sujeito-Verbo-Objeto) e soa exatamente como um paciente brasileiro se expressaria de forma espontânea em um contexto clínico. A escolha do verbo 'representar' para 'depict' também é muito feliz e adequada ao contexto da arteterapia, cumprindo com louvor a Regra 4. | [PhD em A]: A tradução captura a essência da externalização simbólica fundamental à arteterapia. O uso do verbo 'representar' é clinicamente preciso, pois diferencia a mera expressão emocional da capacidade de cri


✅ Sucesso! O arquivo 'CVC_Final_Arteterapia.xlsx' está pronto para download.


In [ ]:
#@title **Listar Modelos Disponíveis (Sintaxe SDK 2.0)**

# Lista todos os modelos acessíveis pela sua chave
print(f"{'ID do Modelo':<50} | {'Ações Suportadas'}")
print("-" * 80)

for model in client.models.list():
    # Filtramos apenas os que permitem gerar conteúdo para facilitar a leitura
    if 'generateContent' in model.supported_actions:
        print(f"{model.name:<50} | {model.supported_actions}")

ID do Modelo                                       | Ações Suportadas
--------------------------------------------------------------------------------
models/gemini-2.5-flash                            | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro                              | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash                            | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001                        | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001                   | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite                       | ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts    